In [ ]:
import os
from PIL import Image
from openai import OpenAI
import torch
from diffusers import StableDiffusionPipeline

# 1. Yerel Ollama (Llama 3.2) İstemcisi
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

try:
    user_prompt = 'batman'
    
    # -------------------------------------------------------------------------
    # BÖLÜM 1: Llama 3.2 ile Prompt Süsleme (Tamamen Yerel)
    # -------------------------------------------------------------------------
    print("[İŞLEM 1] Llama 3.2 ile görsel promptu optimize ediliyor...")
    
    response = client.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {
                "role": "system", 
                "content": (
                    "You are an expert prompt engineer for Stable Diffusion. "
                    "Expand the given prompt with artistic details, style, lighting, and colors. "
                    "Keep your response concise and separated by commas. "
                    "Do not write any introduction or explanation."
                )
            },
            {"role": "user", "content": user_prompt}
        ]
    )
    
    optimized_prompt = response.choices[0].message.content
    print(f"[LLAMA 3.2 PROMPT]: {optimized_prompt}")

    # -------------------------------------------------------------------------
    # BÖLÜM 2: PyTorch & Diffusers ile Görsel Üretimi (Tamamen Yerel / Offline)
    # -------------------------------------------------------------------------
    print("\n[İŞLEM 2] Stable Diffusion modeli bilgisayar hafızasına yükleniyor...")
    
    # Bilgisayarda NVIDIA ekran kartı (CUDA) aktif mi kontrol et
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Modeli Hugging Face'den lokal belleğe yükleyen pipeline (Boru Hattı)
    # float16/float32 seçimi donanımına göre optimize edilir
    pipe = StableDiffusionPipeline.from_pretrained(
        "stabilityai/stable-diffusion-2-1-base",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)
    
    print(f"[İŞLEM 3] Görsel {device} üzerinde yerel olarak çiziliyor...")
    
    # Görüntüyü yerel işlemcimizle/ekran kartımızla üretiyoruz (İnternet bağlantısı YOK)
    # SD 2.1 için optimum çözünürlük 512x512 pikselliktir
    local_image = pipe(
        prompt=optimized_prompt,
        height=512,
        width=512
    ).images[0]

    # -------------------------------------------------------------------------
    # BÖLÜM 3: Dosya Sistemi Operasyonları (Kaydetme ve Gösterme)
    # -------------------------------------------------------------------------
    image_dir = os.path.join(os.curdir, 'images')
    if not os.path.isdir(image_dir):
        os.mkdir(image_dir)

    image_path = os.path.join(image_dir, 'fully-local-image.png')

    # Resmi doğrudan diske kaydediyoruz (İndirme/Download adımı tamamen kalktı!)
    local_image.save(image_path)
    print(f"\n[BAŞARI] Görsel tamamen yerel kaynaklarla üretildi ve kaydedildi: {image_path}")

    # Resmi ekranda göster
    image = Image.open(image_path)
    image.show()

except Exception as err:
    print(f"Sistem Hatası: {err}")